In [0]:
"""
Session Manager
===============

Manages conversation sessions and stores conversation history.

Key concepts:
1. Create sessions (start new conversation)
2. Add messages (store user/agent messages)
3. Retrieve history (get conversation context)
4. End sessions (close conversation)
"""

import pandas as pd
from datetime import datetime
from typing import List, Dict, Optional
import uuid

# ============================================
# 1. SESSION STORAGE (In-Memory + Spark Table)
# ============================================

# Initialize storage in Databricks
print("="*60)
print("SESSION MANAGER: Conversation Memory")
print("="*60)

# Create sessions table
print("\nStep 1: Setting up session storage...")

spark.sql("USE banking_agent_db")

# Create sessions table if not exists
spark.sql("""
CREATE TABLE IF NOT EXISTS sessions (
    session_id STRING,
    user_id STRING,
    created_at TIMESTAMP,
    status STRING,
    last_updated TIMESTAMP
)
USING DELTA
""")

# Create messages table if not exists
spark.sql("""
CREATE TABLE IF NOT EXISTS session_messages (
    message_id STRING,
    session_id STRING,
    role STRING,
    content STRING,
    created_at TIMESTAMP
)
USING DELTA
""")

print("✓ Tables created (sessions, session_messages)")

# ============================================
# 2. SESSION CLASS
# ============================================

class SessionManager:
    """
    Manages user sessions and conversation history.
    
    Example:
        manager = SessionManager()
        
        # Start new session
        session_id = manager.create_session(user_id="user_123")
        
        # Add messages
        manager.add_message(session_id, role="user", 
                          content="What's my balance?")
        manager.add_message(session_id, role="agent",
                          content="Your balance is $5,000")
        
        # Get history
        history = manager.get_history(session_id)
        # Returns: [
        #   {"role": "user", "content": "What's my balance?"},
        #   {"role": "agent", "content": "Your balance is $5,000"}
        # ]
        
        # End session
        manager.end_session(session_id)
    """
    
    def create_session(self, user_id: str) -> str:
        """
        Create a new session for a user.
        
        Args:
            user_id: Unique user identifier
            
        Returns:
            session_id: Unique session identifier
        """
        session_id = f"sess_{uuid.uuid4().hex[:12]}"
        now = datetime.now()
        
        # Store session in Databricks
        session_data = spark.createDataFrame([
            (session_id, user_id, now, "active", now)
        ], ["session_id", "user_id", "created_at", "status", "last_updated"])
        
        session_data.write.mode("append").saveAsTable("sessions")
        
        print(f"✓ Session created: {session_id}")
        print(f"  User: {user_id}")
        print(f"  Time: {now}")
        
        return session_id
    
    def add_message(self, session_id: str, role: str, content: str) -> None:
        """
        Add a message to session history.
        
        Args:
            session_id: Session identifier
            role: "user" or "agent"
            content: Message text
            
        Why store messages?
        - Conversation history for context
        - Audit trail for compliance
        - Analytics on agent performance
        """
        message_id = f"msg_{uuid.uuid4().hex[:12]}"
        now = datetime.now()
        
        # Store message in Databricks
        message_data = spark.createDataFrame([
            (message_id, session_id, role, content, now)
        ], ["message_id", "session_id", "role", "content", "created_at"])
        
        message_data.write.mode("append").saveAsTable("session_messages")
        
        print(f"  [{role.upper()}] {content[:50]}...")
    
    def get_history(self, session_id: str) -> List[Dict]:
        """
        Retrieve conversation history for a session.
        
        Args:
            session_id: Session identifier
            
        Returns:
            List of messages in order:
            [
                {"role": "user", "content": "Question?"},
                {"role": "agent", "content": "Answer"},
                ...
            ]
        """
        try:
            # Query messages from Databricks
            messages = spark.sql(f"""
            SELECT role, content, created_at
            FROM session_messages
            WHERE session_id = '{session_id}'
            ORDER BY created_at ASC
            """).collect()
            
            # Convert to list of dicts
            history = [
                {
                    "role": row["role"],
                    "content": row["content"],
                    "timestamp": str(row["created_at"])
                }
                for row in messages
            ]
            
            return history
            
        except Exception as e:
            print(f"Note: {e}")
            return []
    
    def get_session_summary(self, session_id: str) -> Dict:
        """
        Get summary of a session.
        
        Returns:
            {
                "session_id": "sess_xxx",
                "message_count": 4,
                "turns": 2,
                "status": "active"
            }
        """
        try:
            # Get session info
            session = spark.sql(f"""
            SELECT * FROM sessions WHERE session_id = '{session_id}'
            """).collect()
            
            if not session:
                return {}
            
            # Count messages
            message_count = spark.sql(f"""
            SELECT COUNT(*) as count FROM session_messages 
            WHERE session_id = '{session_id}'
            """).collect()[0]["count"]
            
            # Turns = user messages (agent turns are responses)
            turns = spark.sql(f"""
            SELECT COUNT(*) as count FROM session_messages 
            WHERE session_id = '{session_id}' AND role = 'user'
            """).collect()[0]["count"]
            
            row = session[0]
            
            return {
                "session_id": session_id,
                "user_id": row["user_id"],
                "created_at": str(row["created_at"]),
                "status": row["status"],
                "message_count": message_count,
                "turns": turns
            }
            
        except Exception as e:
            print(f"Note: {e}")
            return {}
    
    def end_session(self, session_id: str) -> None:
        """
        End a session (mark as completed).
        
        Args:
            session_id: Session identifier
        """
        try:
            spark.sql(f"""
            UPDATE sessions 
            SET status = 'completed', last_updated = NOW()
            WHERE session_id = '{session_id}'
            """)
            
            print(f"✓ Session ended: {session_id}")
            
        except Exception as e:
            print(f"Note: {e}")
    
    def list_user_sessions(self, user_id: str) -> List[Dict]:
        """
        List all sessions for a user.
        
        Args:
            user_id: User identifier
            
        Returns:
            List of sessions
        """
        try:
            sessions = spark.sql(f"""
            SELECT session_id, created_at, status
            FROM sessions
            WHERE user_id = '{user_id}'
            ORDER BY created_at DESC
            """).collect()
            
            return [
                {
                    "session_id": row["session_id"],
                    "created_at": str(row["created_at"]),
                    "status": row["status"]
                }
                for row in sessions
            ]
            
        except Exception as e:
            print(f"Note: {e}")
            return []

# ============================================
# 3. TEST SESSION MANAGER
# ============================================

print("\n" + "="*60)
print("TESTING SESSION MANAGER")
print("="*60)

manager = SessionManager()

# Test 1: Create session
print("\nTest 1: Create Session")
print("-"*60)
session_id = manager.create_session(user_id="user_123")

# Test 2: Add messages (multi-turn conversation)
print("\nTest 2: Multi-Turn Conversation")
print("-"*60)

print("\nTurn 1:")
manager.add_message(session_id, "user", "What's my account balance?")
manager.add_message(session_id, "agent", "Your current balance is $5,000")

print("\nTurn 2:")
manager.add_message(session_id, "user", "Can I transfer $3,000 to savings?")
manager.add_message(session_id, "agent", "Yes, $3,000 is within your $5,000 balance. Transfer approved.")

print("\nTurn 3:")
manager.add_message(session_id, "user", "How long will the transfer take?")
manager.add_message(session_id, "agent", "Your $3,000 transfer will complete in 1 business day.")

# Test 3: Retrieve history
print("\n" + "-"*60)
print("Test 3: Retrieve Conversation History")
print("-"*60)

history = manager.get_history(session_id)
print(f"\n✓ Retrieved {len(history)} messages:\n")

for i, msg in enumerate(history, 1):
    role = msg["role"].upper()
    content = msg["content"]
    print(f"{i}. [{role}] {content}")

# Test 4: Get session summary
print("\n" + "-"*60)
print("Test 4: Session Summary")
print("-"*60)

summary = manager.get_session_summary(session_id)
if summary:
    print(f"\n✓ Session Summary:")
    print(f"  Session ID: {summary['session_id']}")
    print(f"  User ID: {summary['user_id']}")
    print(f"  Status: {summary['status']}")
    print(f"  Messages: {summary['message_count']}")
    print(f"  Turns: {summary['turns']}")

# Test 5: End session
print("\n" + "-"*60)
print("Test 5: End Session")
print("-"*60)

manager.end_session(session_id)

print("\n" + "="*60)
print("✓ SESSION MANAGER READY")
print("="*60)

# Export for Phase 3
session_manager = manager

print(f"\n✓ Exported for Phase 3:")
print(f"  - session_manager (SessionManager instance)")
print(f"  - sessions table (Databricks)")
print(f"  - session_messages table (Databricks)")